# Bước 3: Tiền xử lý dữ liệu và Kỹ nghệ đặc trưng (Preprocessing & Feature Engineering)

Notebook này thực hiện mã hóa các cột dữ liệu phân loại sang dạng số (nhị phân, thứ tự và one-hot) và lưu trữ tập dữ liệu sạch chuẩn bị cho việc training.

In [7]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score, classification_report, confusion_matrix

print("Imported necessary libraries successfully!")

Imported necessary libraries successfully!


In [8]:
# Tìm đường dẫn tệp raw data bằng cách leo lên các thư mục cha
data_path = 'data/raw/Student_Performance.csv'
for _ in range(5):
    if os.path.exists(data_path):
        break
    data_path = os.path.join('..', data_path)

df_raw = pd.read_csv(data_path)
df_raw.columns = df_raw.columns.str.strip()


## 2. Tiền xử lý đặc trưng (Feature Engineering)

Chuyển đổi dữ liệu chữ thành dạng nhị phân, mã hóa thứ tự và One-Hot Encoding.

In [ ]:
df_encoded = df_raw.copy()

# 1. Mã hóa nhị phân
binary_cols = {
    'school_type': {'public': 0, 'private': 1},
    'internet_access': {'no': 0, 'yes': 1},
    'extra_activities': {'no': 0, 'yes': 1}
}

for col, mapping in binary_cols.items():
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].map(mapping)

# 2. Mã hóa Ordinal cho parent_education
parent_edu_map = {
    'no formal': 0,
    'high school': 1,
    'diploma': 2,
    'graduate': 3,
    'post graduate': 4,
    'phd': 5
}
df_encoded['parent_education'] = df_encoded['parent_education'].str.lower().str.strip().map(parent_edu_map)

# 3. One-Hot Encoding cho các cột nominal
nominal_cols = ['gender', 'travel_time', 'study_method']
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols)

# 4. Xác định danh sách đặc trưng đầu vào X theo đúng DataPreprocessor
objective_features = [
    'age', 'study_hours', 'attendance_percentage',
    'math_score', 'science_score', 'english_score',
    'parent_education',
    'gender_male', 'gender_female', 'gender_other',
    'travel_time_<15 min', 'travel_time_15-30 min', 'travel_time_30-60 min', 'travel_time_>60 min',
    'study_method_notes', 'study_method_textbook', 'study_method_group study', 
    'study_method_coaching', 'study_method_mixed', 'study_method_online videos'
]

# Kiểm tra an toàn: đảm bảo overall_score không lọt vào tập đặc trưng
assert 'overall_score' not in objective_features, "overall_score bi ro ri du lieu, khong duoc dua vao feature set!"

# Tự động tính đường dẫn lưu dữ liệu processed tương ứng
project_root = os.path.dirname(os.path.dirname(os.path.dirname(data_path)))
output_path = os.path.join(project_root, 'data', 'processed', 'Student_Performance_processed.csv') if project_root else os.path.join('data', 'processed', 'Student_Performance_processed.csv')

df_encoded.to_csv(output_path, index=False)
print(f"Objective features count: {len(objective_features)}")
print(f"Saved preprocessed data successfully to: {output_path}")


Objective features count: 17
Saved preprocessed data successfully to: ..\data\processed\Student_Performance_processed.csv
